# MLX Solar Radiation Prediction - Grandmaster Stacking Pipeline

This notebook implements the full leakage-aware solar radiation pipeline:

- strict `UNIXTime` sorting
- cyclic temporal features
- dew point / moisture boundary features
- clear-sky solar physics baseline for Kolkata latitude
- shifted lag and rolling sensor dynamics
- `log1p` target transform
- LightGBM + XGBoost + CatBoost base models
- TimeSeriesSplit out-of-fold predictions
- Ridge stacking meta-learner
- physical post-processing and `submission.csv`

Default mode is a fast smoke run. Change `RUN_MODE` to `"tuned"` or `"full"` in the config cell for stronger submissions.

## 0. Optional Package Install

Run this cell only on a fresh environment where the packages are missing.

In [ ]:
# Kaggle usually has some of these installed, but this makes the notebook self-contained.
%pip install -q catboost optuna lightgbm xgboost


## 1. Imports And Configuration

In [ ]:
import json
import os
from pathlib import Path
from typing import Callable

# CatBoost imports polars in recent versions; this avoids a local CPU feature check issue on some machines.
os.environ.setdefault("POLARS_SKIP_CPU_CHECK", "1")

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
LATITUDE_DEGREES = 22.5
SOLAR_CONSTANT = 1361.0
CLIP_MARGIN = 0.0


def find_input_file(filename: str) -> str:
    search_roots = [Path('/kaggle/input'), Path('.')]
    for root in search_roots:
        if root.exists():
            matches = sorted(root.rglob(filename))
            if matches:
                return str(matches[0])
    raise FileNotFoundError(f"Could not find {filename}. Check that it was uploaded to Kaggle input.")


TRAIN_PATH = find_input_file("train_df_1.csv")
TEST_PATH = find_input_file("test_df_1.csv")
OUTPUT_PATH = "/kaggle/working/submission.csv" if Path("/kaggle/working").exists() else "submission.csv"
FEATURES_OUT = "/kaggle/working/features_used.json" if Path("/kaggle/working").exists() else "features_used.json"

# Modes:
#   quick -> fast smoke test, no Optuna tuning
#   tuned -> 10 Optuna trials per model
#   full  -> 100 Optuna trials per model
RUN_MODE = "full"

MODE_CONFIG = {
    "quick": {"splits": 3, "trials": 0, "early_stopping_rounds": 20},
    "tuned": {"splits": 5, "trials": 10, "early_stopping_rounds": 50},
    "full": {"splits": 5, "trials": 100, "early_stopping_rounds": 50},
}

cfg = MODE_CONFIG[RUN_MODE]
N_SPLITS = cfg["splits"]
N_TRIALS = cfg["trials"]
TUNE_SPLITS = min(3, N_SPLITS)
EARLY_STOPPING_ROUNDS = cfg["early_stopping_rounds"]
RIDGE_ALPHA = 1.0

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Mode: {RUN_MODE} | splits={N_SPLITS} | tune_splits={TUNE_SPLITS} | trials/model={N_TRIALS} | output={OUTPUT_PATH}")


## 2. Load, Sort, And Validate Data

In [ ]:
REQUIRED_TRAIN_COLUMNS = {
    "ID", "UNIXTime", "Radiation", "Temperature", "Pressure",
    "Humidity", "WindDirection", "Speed",
}
REQUIRED_TEST_COLUMNS = REQUIRED_TRAIN_COLUMNS - {"Radiation"}


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {
        "WindDirection(Degrees)": "WindDirection",
        "WindDirectionDegrees": "WindDirection",
        "Wind Direction": "WindDirection",
    }
    return df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})


def validate_columns(df: pd.DataFrame, required: set[str], name: str) -> None:
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")


def load_sorted_data(train_path: str, test_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    train = normalize_columns(pd.read_csv(train_path))
    test = normalize_columns(pd.read_csv(test_path))
    test["_original_order"] = np.arange(len(test))

    validate_columns(train, REQUIRED_TRAIN_COLUMNS, "train")
    validate_columns(test, REQUIRED_TEST_COLUMNS, "test")

    train = train.sort_values("UNIXTime", kind="mergesort").reset_index(drop=True)
    test = test.sort_values("UNIXTime", kind="mergesort").reset_index(drop=True)
    return train, test


train_raw, test_raw = load_sorted_data(TRAIN_PATH, TEST_PATH)
print("Train shape:", train_raw.shape)
print("Test shape:", test_raw.shape)
print("Train time range:", pd.to_datetime(train_raw.UNIXTime, unit="s").min(), "to", pd.to_datetime(train_raw.UNIXTime, unit="s").max())
print("Test time range:", pd.to_datetime(test_raw.UNIXTime, unit="s").min(), "to", pd.to_datetime(test_raw.UNIXTime, unit="s").max())
print("Negative train target count:", int((train_raw["Radiation"] < 0).sum()))

## 3. Feature Engineering Engine

In [ ]:
def time_string_to_minutes(series: pd.Series) -> pd.Series:
    delta = pd.to_timedelta(series, errors="coerce")
    return delta.dt.total_seconds() / 60.0


def local_hour_from_time(df: pd.DataFrame) -> pd.Series:
    if "Time" in df.columns:
        parts = df["Time"].astype(str).str.split(":", expand=True)
        if parts.shape[1] >= 3:
            parts = parts.iloc[:, :3].apply(pd.to_numeric, errors="coerce")
            hour = parts[0] + parts[1] / 60.0 + parts[2] / 3600.0
            return hour.fillna((pd.to_numeric(df["UNIXTime"], errors="coerce") % 86400) / 3600.0)
    return (pd.to_numeric(df["UNIXTime"], errors="coerce") % 86400) / 3600.0


def local_day_of_year(df: pd.DataFrame) -> pd.Series:
    unix_day = (pd.to_numeric(df["UNIXTime"], errors="coerce") // 86400) % 365
    if "Data" in df.columns:
        parsed = pd.to_datetime(df["Data"], format="%d-%m-%Y", errors="coerce")
        return parsed.dt.dayofyear.fillna(unix_day)
    return unix_day


def engineer_static_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    hour = local_hour_from_time(df)
    day_of_year = local_day_of_year(df)

    df["hour"] = hour
    df["dayofyear"] = day_of_year
    if "Data" in df.columns:
        local_date = pd.to_datetime(df["Data"], format="%d-%m-%Y", errors="coerce")
        df["month"] = local_date.dt.month.fillna(pd.to_datetime(df["UNIXTime"], unit="s", errors="coerce").dt.month)
    else:
        df["month"] = pd.to_datetime(df["UNIXTime"], unit="s", errors="coerce").dt.month
    df["minutes_since_midnight"] = hour * 60.0

    df["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    df["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    df["doy_sin"] = np.sin(2.0 * np.pi * day_of_year / 365.0)
    df["doy_cos"] = np.cos(2.0 * np.pi * day_of_year / 365.0)
    df["month_sin"] = np.sin(2.0 * np.pi * df["month"] / 12.0)
    df["month_cos"] = np.cos(2.0 * np.pi * df["month"] / 12.0)

    df["is_dawn"] = ((hour >= 5.0) & (hour < 8.0)).astype(int)
    df["is_morning"] = ((hour >= 8.0) & (hour < 11.0)).astype(int)
    df["is_solar_noon"] = ((hour >= 11.0) & (hour < 14.0)).astype(int)
    df["is_afternoon"] = ((hour >= 14.0) & (hour < 17.0)).astype(int)
    df["is_dusk"] = ((hour >= 17.0) & (hour < 20.0)).astype(int)
    df["is_night"] = ((hour < 5.0) | (hour >= 20.0)).astype(int)

    lat_rad = np.radians(LATITUDE_DEGREES)
    declination = 23.45 * np.sin(np.radians((360.0 / 365.0) * (day_of_year - 81.0)))
    hour_angle = 15.0 * (hour - 12.0)
    cos_zenith = (
        np.sin(lat_rad) * np.sin(np.radians(declination))
        + np.cos(lat_rad) * np.cos(np.radians(declination)) * np.cos(np.radians(hour_angle))
    )
    df["cos_zenith"] = np.clip(cos_zenith, 0.0, 1.0)
    df["clear_sky_rad"] = SOLAR_CONSTANT * df["cos_zenith"]
    df["clear_sky"] = df["clear_sky_rad"]
    df["solar_zenith_angle"] = np.degrees(np.arccos(df["cos_zenith"].clip(0.0, 1.0)))

    temp = df["Temperature"]
    humidity = df["Humidity"]
    alpha = ((17.27 * temp) / (237.7 + temp)) + np.log((humidity / 100.0).clip(lower=1e-6))
    df["dew_point"] = (237.7 * alpha) / (17.27 - alpha)
    df["dew_point_gap"] = temp - df["dew_point"]

    wind_rad = np.radians(df["WindDirection"])
    df["u_wind"] = df["Speed"] * np.cos(wind_rad)
    df["v_wind"] = df["Speed"] * np.sin(wind_rad)
    df["wind_speed_sq"] = df["Speed"] ** 2

    df["temp_humidity_ratio"] = temp / (humidity + 1.0)
    df["temp_humidity_product"] = temp * humidity
    df["temp_pressure"] = temp * df["Pressure"]
    df["humidity_pressure"] = humidity * df["Pressure"]
    df["temp_clear_sky"] = temp * df["clear_sky_rad"]
    df["temp_sq"] = temp ** 2
    df["humidity_sq"] = humidity ** 2
    df["pressure_sq"] = df["Pressure"] ** 2

    if "TimeSunRise" in df.columns:
        df["sunrise_minutes"] = time_string_to_minutes(df["TimeSunRise"])
    if "TimeSunSet" in df.columns:
        df["sunset_minutes"] = time_string_to_minutes(df["TimeSunSet"])
    if {"sunrise_minutes", "sunset_minutes"}.issubset(df.columns):
        df["daylight_minutes"] = df["sunset_minutes"] - df["sunrise_minutes"]
        df["minutes_after_sunrise"] = df["minutes_since_midnight"] - df["sunrise_minutes"]
        df["minutes_before_sunset"] = df["sunset_minutes"] - df["minutes_since_midnight"]

    return df


def add_shifted_dynamics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Strictly shifted features: no current-row self-peeking.
    for col in ["Temperature", "Humidity"]:
        df[f"{col}_lag_1"] = df[col].shift(1).bfill()

    for col in ["Temperature", "Humidity", "Pressure", "Speed"]:
        for lag in (1, 2, 3, 5):
            df[f"{col}_lag_{lag}"] = df[col].shift(lag)
        shifted = df[col].shift(1)
        for window in (3, 6, 12):
            df[f"{col}_rmean_{window}"] = shifted.rolling(window=window, min_periods=1).mean()
            df[f"{col}_rstd_{window}"] = shifted.rolling(window=window, min_periods=2).std()

    if "Radiation" in df.columns:
        if "_is_test" in df.columns:
            known_radiation = df["Radiation"].where(df["_is_test"].eq(0))
        else:
            known_radiation = df["Radiation"]
        known_radiation = known_radiation.clip(lower=0.0).ffill()
        for lag in (1, 2, 3, 5):
            df[f"known_radiation_lag_{lag}"] = known_radiation.shift(lag)
        shifted_radiation = known_radiation.shift(1)
        for window in (3, 6):
            df[f"known_radiation_rmean_{window}"] = shifted_radiation.rolling(window=window, min_periods=1).mean()
            df[f"known_radiation_rstd_{window}"] = shifted_radiation.rolling(window=window, min_periods=2).std()

    df["pressure_diff_1"] = df["Pressure"].diff(1).fillna(0.0)
    df["pressure_diff_3"] = df["Pressure"].diff(3).fillna(0.0)
    df["temperature_diff_1"] = df["Temperature"].diff(1)
    df["humidity_diff_1"] = df["Humidity"].diff(1)
    df["speed_diff_1"] = df["Speed"].diff(1)

    return df


def build_feature_frames(train: pd.DataFrame, test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train = train.copy()
    test = test.copy()
    train["_is_test"] = 0
    test["_is_test"] = 1

    combined = pd.concat([train, test], axis=0, ignore_index=True, sort=False)
    combined = combined.sort_values("UNIXTime", kind="mergesort").reset_index(drop=True)
    combined = engineer_static_features(combined)
    combined = add_shifted_dynamics(combined)

    train_fe = combined[combined["_is_test"] == 0].copy().reset_index(drop=True)
    test_fe = combined[combined["_is_test"] == 1].copy().reset_index(drop=True)
    return train_fe, test_fe


def make_matrices(
    train: pd.DataFrame, test: pd.DataFrame, features_out: str
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, np.ndarray, pd.DataFrame, list[str]]:
    drop_cols = {
        "ID", "UNIXTime", "Data", "Time", "Radiation", "TimeSunRise", "TimeSunSet",
        "_is_test", "_original_order",
    }
    feature_columns = [
        col for col in train.columns
        if col not in drop_cols and pd.api.types.is_numeric_dtype(train[col])
    ]

    X_train = train[feature_columns]
    X_test = test[feature_columns]

    # Negative radiation values are non-physical sensor noise and break log1p below -1.
    y_train = np.log1p(train["Radiation"].clip(lower=0.0))
    clear_sky_test = test["clear_sky_rad"].to_numpy()

    Path(features_out).write_text(json.dumps(feature_columns, indent=2), encoding="utf-8")
    return X_train, y_train, X_test, clear_sky_test, test[["_original_order", "ID"]].copy(), feature_columns

## 4. Build Training Matrices

In [ ]:
train_fe, test_fe = build_feature_frames(train_raw, test_raw)
X_train, y_train, X_test, clear_sky_test, test_ids, feature_columns = make_matrices(
    train_fe, test_fe, FEATURES_OUT
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Features:", len(feature_columns))
print("Feature list saved to:", FEATURES_OUT)
X_train.head()

## 5. Model Factories, Tuning, And OOF Helpers

In [ ]:
def rmse(y_true: np.ndarray | pd.Series, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def lgb_default_params(quick: bool) -> dict:
    return {
        "n_estimators": 120 if quick else 900,
        "learning_rate": 0.07 if quick else 0.03,
        "max_depth": 6 if quick else -1,
        "num_leaves": 31 if quick else 63,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.05,
        "reg_lambda": 1.0,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
    }


def xgb_default_params(quick: bool) -> dict:
    return {
        "n_estimators": 120 if quick else 900,
        "learning_rate": 0.07 if quick else 0.03,
        "max_depth": 4 if quick else 6,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.05,
        "reg_lambda": 1.0,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": 0,
    }


def cat_default_params(quick: bool) -> dict:
    return {
        "iterations": 120 if quick else 900,
        "learning_rate": 0.07 if quick else 0.03,
        "depth": 5 if quick else 7,
        "l2_leaf_reg": 5.0,
        "loss_function": "RMSE",
        "random_seed": RANDOM_STATE,
        "thread_count": -1,
        "verbose": 0,
    }


def fit_model(name: str, model, X_tr, y_tr, X_val, y_val, early_stopping_rounds: int):
    if name == "lgb":
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
        )
    elif name == "xgb":
        model.set_params(early_stopping_rounds=early_stopping_rounds)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    elif name == "cat":
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            early_stopping_rounds=early_stopping_rounds,
            use_best_model=True,
        )
    else:
        raise ValueError(f"Unknown model name: {name}")
    return model


def tune_lightgbm(X, y, splitter, trials: int, quick: bool) -> dict:
    if trials <= 0:
        return lgb_default_params(quick)

    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "num_leaves": trial.suggest_int("num_leaves", 20, 256),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "verbose": -1,
        }
        scores = []
        for fold, (tr_idx, val_idx) in enumerate(splitter.split(X)):
            model = lgb.LGBMRegressor(**params)
            fit_model("lgb", model, X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[val_idx], y.iloc[val_idx], 50)
            scores.append(rmse(y.iloc[val_idx], model.predict(X.iloc[val_idx])))
            trial.report(float(np.mean(scores)), step=fold)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=trials, show_progress_bar=False)
    return {**lgb_default_params(quick), **study.best_params}


def tune_xgboost(X, y, splitter, trials: int, quick: bool) -> dict:
    if trials <= 0:
        return xgb_default_params(quick)

    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "verbosity": 0,
        }
        scores = []
        for fold, (tr_idx, val_idx) in enumerate(splitter.split(X)):
            model = xgb.XGBRegressor(**params)
            fit_model("xgb", model, X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[val_idx], y.iloc[val_idx], 50)
            scores.append(rmse(y.iloc[val_idx], model.predict(X.iloc[val_idx])))
            trial.report(float(np.mean(scores)), step=fold)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=trials, show_progress_bar=False)
    return {**xgb_default_params(quick), **study.best_params}


def tune_catboost(X, y, splitter, trials: int, quick: bool) -> dict:
    if trials <= 0:
        return cat_default_params(quick)

    def objective(trial: optuna.Trial) -> float:
        params = {
            "iterations": trial.suggest_int("iterations", 300, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
            "loss_function": "RMSE",
            "random_seed": RANDOM_STATE,
            "thread_count": -1,
            "verbose": 0,
        }
        scores = []
        for fold, (tr_idx, val_idx) in enumerate(splitter.split(X)):
            model = CatBoostRegressor(**params)
            fit_model("cat", model, X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[val_idx], y.iloc[val_idx], 50)
            scores.append(rmse(y.iloc[val_idx], model.predict(X.iloc[val_idx])))
            trial.report(float(np.mean(scores)), step=fold)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=trials, show_progress_bar=False)
    return {**cat_default_params(quick), **study.best_params}


def make_oof_and_test_predictions(name, model_factory: Callable[[], object], X, y, X_test, splitter, early_stopping_rounds):
    oof = np.full(len(X), np.nan)
    test_predictions = np.zeros(len(X_test), dtype=float)

    for fold, (tr_idx, val_idx) in enumerate(splitter.split(X), start=1):
        model = model_factory()
        fit_model(
            name, model,
            X.iloc[tr_idx], y.iloc[tr_idx],
            X.iloc[val_idx], y.iloc[val_idx],
            early_stopping_rounds,
        )
        oof[val_idx] = model.predict(X.iloc[val_idx])
        test_predictions += model.predict(X_test) / splitter.n_splits
        print(f"{name.upper()} fold {fold}: log RMSE {rmse(y.iloc[val_idx], oof[val_idx]):.5f}")

    valid = np.isfinite(oof)
    print(f"{name.upper()} OOF log RMSE: {rmse(y[valid], oof[valid]):.5f} on {valid.sum()} rows")
    return oof, test_predictions

## 6. Tune Base Learners And Generate OOF Predictions

In [ ]:
splitter = TimeSeriesSplit(n_splits=N_SPLITS)
tune_splitter = TimeSeriesSplit(n_splits=TUNE_SPLITS)
quick = RUN_MODE == "quick"

if N_TRIALS:
    print(f"Optuna search uses {TUNE_SPLITS} folds; final OOF uses {N_SPLITS} folds.")

print("Tuning LightGBM..." if N_TRIALS else "Using default LightGBM params...")
lgb_params = tune_lightgbm(X_train, y_train, tune_splitter, N_TRIALS, quick)

print("Tuning XGBoost..." if N_TRIALS else "Using default XGBoost params...")
xgb_params = tune_xgboost(X_train, y_train, tune_splitter, N_TRIALS, quick)

print("Tuning CatBoost..." if N_TRIALS else "Using default CatBoost params...")
cat_params = tune_catboost(X_train, y_train, tune_splitter, N_TRIALS, quick)

print("Generating OOF and averaged test predictions...")
lgb_oof, lgb_test = make_oof_and_test_predictions(
    "lgb", lambda: lgb.LGBMRegressor(**lgb_params),
    X_train, y_train, X_test, splitter, EARLY_STOPPING_ROUNDS,
)
xgb_oof, xgb_test = make_oof_and_test_predictions(
    "xgb", lambda: xgb.XGBRegressor(**xgb_params),
    X_train, y_train, X_test, splitter, EARLY_STOPPING_ROUNDS,
)
cat_oof, cat_test = make_oof_and_test_predictions(
    "cat", lambda: CatBoostRegressor(**cat_params),
    X_train, y_train, X_test, splitter, EARLY_STOPPING_ROUNDS,
)

## 7. Ridge Stacking, Physical Clipping, And Submission

In [ ]:
meta_train = np.column_stack([lgb_oof, xgb_oof, cat_oof])
meta_test = np.column_stack([lgb_test, xgb_test, cat_test])
valid_meta = np.isfinite(meta_train).all(axis=1)

scaler = StandardScaler()
meta_train_scaled = scaler.fit_transform(meta_train[valid_meta])
meta_test_scaled = scaler.transform(meta_test)

meta_model = Ridge(alpha=RIDGE_ALPHA)
meta_model.fit(meta_train_scaled, y_train[valid_meta])
meta_oof = meta_model.predict(meta_train_scaled)

print(f"STACK OOF log RMSE: {rmse(y_train[valid_meta], meta_oof):.5f}")
print("Ridge weights:", dict(zip(["lgb", "xgb", "cat"], meta_model.coef_.round(5))))

final_log_predictions = meta_model.predict(meta_test_scaled)
final_predictions = np.expm1(final_log_predictions)
final_predictions = np.clip(final_predictions, 0.0, clear_sky_test + CLIP_MARGIN)

submission = test_ids.copy()
submission["TARGET"] = final_predictions
submission = submission.sort_values("_original_order", kind="mergesort")[["ID", "TARGET"]]
submission.to_csv(OUTPUT_PATH, index=False)

assert submission["ID"].is_unique
assert submission["TARGET"].notna().all()
assert (submission["TARGET"] >= 0).all()

print(f"Saved {OUTPUT_PATH}")
print(submission["TARGET"].describe())
submission.head()

## 8. Submission Sanity Check

In [ ]:
check = pd.read_csv(OUTPUT_PATH)
print(check.shape)
print("NaNs:", check.isna().sum().to_dict())
print("Negative predictions:", int((check["TARGET"] < 0).sum()))
print("Unique IDs:", check["ID"].is_unique)
check.head(10)